In [1]:
 # Imports and setup
from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd
import plotly.express as px
import dash_leaflet as dl

# Import your CRUD module and class - update the filename & class name if needed
from crud_module import AnimalShelter

In [2]:
# Connect to MongoDB via your CRUD module (AnimalShelter)
db = AnimalShelter()  # pass user/pass inside class if hardcoded, else modify init parameters

# Read all data from MongoDB (empty query returns everything)
all_data = db.read({})

# Convert data to pandas DataFrame
df = pd.DataFrame(all_data)

# Remove MongoDB '_id' field to prevent dash_table errors
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

Connected to MongoDB successfully.


In [3]:
# Encode the Grazioso Salvare logo image to base64
image_filename = 'assets/grazioso_salvare_logo.png'  # Put logo image in 'assets' folder
with open(image_filename, 'rb') as img_file:
    encoded_image = base64.b64encode(img_file.read()).decode()

In [4]:
# Define Rescue Filter options for the radio items filtering
rescue_type_options = [
    {'label': 'Water Rescue', 'value': 'Water Rescue'},
    {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
    {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
    {'label': 'Reset', 'value': 'Reset'}
]

In [5]:
# Initialize the Dash app
app = JupyterDash(__name__)

# Layout of the dashboard
app.layout = html.Div([
    # Logo and Unique Identifier
    html.Div([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image), style={'height': '100px'}),
        html.H5("Dashboard developed by JFabian - 2985375"),
        html.H1("Grazioso Salvare Dog Rescue Dashboard"),
        html.Hr(),
    ], style={'textAlign': 'center'}),
    
    # Rescue Type Filter (radio buttons)
    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='rescue-filter',
            options=rescue_type_options,
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ], style={'padding': '10px 20px'}),
    
    # Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'},
        row_selectable='single',
        selected_rows=[]
    ),
    html.Br(),
    
    # Charts and Map side-by-side
    html.Div([
        html.Div(dcc.Graph(id='pie-chart'), style={'width': '48%', 'display': 'inline-block'}),
        html.Div(id='map-container', children=[], style={'width': '48%', 'display': 'inline-block', 'verticalAlign': 'top'}),
    ]),
])

In [6]:
# Callbacks to update dashboard components
@app.callback(
    Output('datatable-id', 'data'),
    [Input('rescue-filter', 'value')]
)
def update_table(selected_rescue):
    if selected_rescue == 'Reset':
        data = db.read({})
    else:
        data = db.read({"rescue_skills": selected_rescue})
    
    df_filtered = pd.DataFrame(data)
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    return df_filtered.to_dict('records')

In [7]:
@app.callback(
    Output('pie-chart', 'figure'),
    [Input('datatable-id', 'data')]
)
def update_pie_chart(table_data):
    df_chart = pd.DataFrame(table_data)
    if df_chart.empty or 'rescue_skills' not in df_chart.columns:
        return {}
    fig = px.pie(df_chart, names='rescue_skills', title='Rescue Skills Distribution')
    return fig

In [8]:
@app.callback(
    Output('map-container', 'children'),
    [Input('datatable-id', 'data'), Input('datatable-id', 'selected_rows')]
)
def update_map(table_data, selected_rows):
    df_map = pd.DataFrame(table_data)
    if df_map.empty:
        return []
    
    # Default coords to Austin TX if nothing selected
    center = [30.2672, -97.7431]

    if selected_rows and len(selected_rows) > 0:
        row_idx = selected_rows[0]
        if 'latitude' in df_map.columns and 'longitude' in df_map.columns:
            center = [df_map.iloc[row_idx]['latitude'], df_map.iloc[row_idx]['longitude']]
    
    markers = []
    for _, row in df_map.iterrows():
        if 'latitude' in df_map.columns and 'longitude' in df_map.columns:
            markers.append(
                dl.Marker(position=[row['latitude'], row['longitude']], children=[
                    dl.Tooltip(row.get('name', 'No Name')),
                    dl.Popup([
                        html.H4(row.get('name', 'No Name')),
                        html.P(f"Rescue Skill: {row.get('rescue_skills', 'N/A')}"),
                        html.P(f"Age: {row.get('age', 'N/A')}"),
                    ])
                ])
            )
    
    return dl.Map(
        style={'width': '100%', 'height': '500px'},
        center=center,
        zoom=10,
        children=[
            dl.TileLayer(),
            *markers
        ]
    )

In [9]:
# Run the app in JupyterLab

app.run_server(mode='jupyterlab', port=8052, debug=True)